In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/noumanarif.personal@gmail.com/Batch_Processing_Data_Engineering/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-childcompany-bucket/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base_Path:",base_path)
print("Landing_Path:", landing_path)
print("processed_path", processed_path)

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"



so when we read from our data source and if that source is file source like csv then when we read data from there apache spark and databricks will maintain the meta data file that will be hidden. that metadata file will have all the information that which row belongs to which file 

and in our code we have added some column to maintain tracability like we have maintained timestamp , at which time that file was pulled from source and also from metadata folder we have also added file name and file size along with each row.

so that later if any row will fail any data quality check or anything like this will happen then we exactly know from which file row came from , thus builds data_lineage.



In [0]:
df = spark.read.options(header = True, inferSchema = True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name","_metadata.file_size")

print("total_rows:", df.count())
df.show()


In [0]:
# Move files from 'processed' back to 'landing'
for file in dbutils.fs.ls(processed_path):
    dbutils.fs.mv(file.path, f"{landing_path}/{file.name}", True)

In [0]:
df.display(df.limit(15))

In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "True") \
    .mode("append") \
    .saveAsTable(bronze_table)

# Now we have to move files from landing folder to processed folder. why?

You have hit on a brilliant architectural nuance. Your logic is completely correct: if we are keeping the data, the storage of that data is persistent, not transient.

The confusion comes from mixing up the **ingestion process** (which is transient) with the **overall storage zone** (which is persistent).

Here is the exact breakdown of why we use this specific `landing` to `processed` move strategy, and why we don't just dump everything into a persistent folder from the start.

### The "Mailbox" vs. The "Filing Cabinet"

In your S3 bucket, your overarching "Raw Zone" actually contains both a transient layer and a persistent layer:

* **`s3://<project>-raw/landing/` (The Mailbox - Transient):** This folder must act like a transient queue. It only holds the data temporarily until the Spark job picks it up. Once the job runs, this folder must be emptied.
* **`s3://<project>-raw/processed/` (The Filing Cabinet - Persistent):** This folder is your true persistent staging layer. It holds the immutable history of every raw file ever ingested.

### Why not just drop data directly into Persistent Staging?

If you had the external API drop files directly into your persistent `processed/` folder from day one, you would create a massive operational headache for your batch processing pipeline.

Imagine your persistent folder has 3 years of daily CSV files (over 1,000 files). If your pipeline is a standard batch job that runs `spark.read.csv("s3://.../processed/*.csv")`, Spark will read **all 1,000 files every single day** just to process the newest one. This wastes massive amounts of compute power and money.

By splitting the architecture into two folders:

1. **The API drops today's file into `landing/`.**
2. **Spark reads `landing/*.csv`.** It only reads 1 file. Fast and cheap.
3. **Spark moves the file to `processed/`.**

You achieve the best of both worlds: the compute engine gets the efficiency of a **transient** queue (only processing new data), and the business gets the safety of a **persistent** historical archive (never throwing away the source of truth).

### The Exception: When to use purely Persistent Staging

You *can* drop files directly into a persistent folder and never move them, but you have to change your compute engine strategy.

Instead of a standard batch read, you would have to use **Databricks Auto Loader** or **Spark Structured Streaming**. These tools maintain an internal checkpoint (a state file) that remembers exactly which files in the persistent directory have already been read.

For standard batch ELT pipelines (like the Modern Lakehouse project you are building with Airflow and standard PySpark batch reads), the `landing` $\rightarrow$ `processed` move pattern is the most robust way to manage state.

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path, f"{processed_path}/{file_info.name}", True)

    


# Silver_Tranformations

In [0]:
df_orders = spark.sql(f"SELECT * from {bronze_table}")
df_orders.show(5)

In [0]:
# 1. first we will solve null order_quantity column
# so keep only the rows where order quantity is not NULL

df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. second keep customer_ids that are numeric
df_orders = df_orders.withColumn("customer_id", F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id")).otherwise("999999").cast("string"))

# 3. Remove Day name from order_placement_date
# i.e Tuesday, July 01, 2025 -> July 01, 2025

df_orders = df_orders.withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", ""))

# Now parse order placement date using multiple possible formats
df_orders = df_orders.withColumn("order_placement_date", F.coalesce(
    F.try_to_date("order_placement_date", "yyyy/MM/dd"),
    F.try_to_date("order_placement_date", "dd-MM-yyyy"),
    F.try_to_date("order_placement_date", "dd/MM/yyyy"),
    F.try_to_date("order_placement_date", "MMMM dd, yyyy")
))

# lets reomove some duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# convert Product_id to string
df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))


In [0]:
display(df_orders.limit(20))

In [0]:
# check maximum and minimum dates
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
df_orders.display()

In [0]:
# lets fetch product_code before writing 
df_products = spark.table("fmcg.silver.products")
display(df_products.limit(5))

In [0]:
# we want our df_orders to have product_code , in order to do this we have to perform join with
# df_products table

df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])
display(df_joined.limit(5))

In [0]:
# Now we will write it to our silver layer
# if tables does not exist then create it other wise perform UPSERT

if not(spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"))